## Indic Parler-TTS — Text-to-Speech on Colab GPU

Model page: https://huggingface.co/ai4bharat/indic-parler-tts

**Important about the auto-generated snippet:** Parler-TTS is *not* a built-in
Transformers architecture, so `AutoModelForSeq2SeqLM` / `pipeline("text-to-speech")`
fail with `model type 'parler_tts' ... not recognized`. You must use the
`ParlerTTSForConditionalGeneration` class from the `parler-tts` package, which
pins `transformers==4.46.1`. **Do not** run `pip install -U transformers` — a newer
transformers drops the path Parler-TTS needs and breaks the model.

In [9]:
# Install Parler-TTS. This pulls in the exact transformers version it requires
# (4.46.1). If a previous run already upgraded transformers, this downgrades it,
# and you must then RESTART the runtime (Runtime > Restart session) before continuing.
!pip install -q git+https://github.com/huggingface/parler-tts.git
!pip install -q "transformers==4.46.1"  # belt-and-suspenders: force the supported version

# transformers 4.46.1 ships protobuf-generated files that need 'runtime_version'
# (protobuf >= 5.26). Colab pins an old protobuf (4.x) for its bundled TensorFlow, so
# importing transformers.modeling_utils fails with:
#   cannot import name 'runtime_version' from 'google.protobuf'
# Upgrade ONLY protobuf here — do NOT add transformers to this line or it'll un-pin 4.46.1.
!pip install -q -U "protobuf>=5.26"

# Verify + fail loudly if the kernel still holds the old protobuf (then restart the runtime).
import importlib.metadata as _md
print("protobuf:", _md.version("protobuf"))   # must be >= 5.26
from google.protobuf import runtime_version    # raises if old protobuf is still loaded → restart session

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.1/327.1 kB 10.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
descript-audiotools 0.7.4 requires protobuf!=4.24.0,<5.0.0,>=3.19.6, but you have protobuf 7.35.1 which is incompatible.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.35.1 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 7.35.1 which is incompatible.
google-cloud-discoveryengine 0.13.12 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have protobuf 7.35.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!

### Authenticate (gated model)
`ai4bharat/indic-parler-tts` is gated. Request access on the model page, then log in
below with a token from https://huggingface.co/settings/tokens (or set the `HF_TOKEN`
Colab secret).

In [10]:
from huggingface_hub import login
login()  # paste an HF token that has access to ai4bharat/indic-parler-tts

### Load the model and the two tokenizers
Indic Parler-TTS uses **two** tokenizers: one for the *prompt* (the text to speak)
and one for the *description* (a natural-language description of the desired voice).

In [11]:
import torch
from parler_tts import ParlerTTSForConditionalGeneration
from transformers import AutoTokenizer

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

model = ParlerTTSForConditionalGeneration.from_pretrained(
    "ai4bharat/indic-parler-tts"
).to(device)

tokenizer = AutoTokenizer.from_pretrained("ai4bharat/indic-parler-tts")
description_tokenizer = AutoTokenizer.from_pretrained(model.config.text_encoder._name_or_path)

print("Sampling rate:", model.config.sampling_rate)

Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.75G [00:00<?, ?B/s]

  "_name_or_path": "google/flan-t5-large",
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 2816,
  "d_kv": 64,
  "d_model": 1024,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 24,
  "num_heads": 16,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "transformers_version": "4.46.1",
  "use_cache": true,
  "vocab_size": 32128
}

  "_name_or_path": "ylacombe/dac_44khz",
  "architectures": [
    "DacModel"
  ],
  "codebook_dim": 8,
  "codebook_loss_weight": 1.0,
  "codebook_size": 1024,
  "commitment_loss_weight": 0.25,
  "decoder_hidden_si

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Sampling rate: 44100


### Generate speech and play it

In [12]:
from IPython.display import Audio

# Text to speak (here: Hindi) and a description of the voice/style.
prompt = "नमस्ते, आप कैसे हैं?"
description = (
    "A female speaker delivers a clear and expressive speech in Hindi, "
    "speaking at a moderate pace in a quiet environment."
)

description_inputs = description_tokenizer(description, return_tensors="pt").to(device)
prompt_inputs = tokenizer(prompt, return_tensors="pt").to(device)

print("Generating audio...")
with torch.no_grad():
    generation = model.generate(
        input_ids=description_inputs.input_ids,
        attention_mask=description_inputs.attention_mask,
        prompt_input_ids=prompt_inputs.input_ids,
        prompt_attention_mask=prompt_inputs.attention_mask,
    )

audio_arr = generation.cpu().numpy().squeeze()
Audio(audio_arr, rate=model.config.sampling_rate)

Generating audio...


### Optional: save to a WAV file

In [13]:
import soundfile as sf
sf.write("indic_tts_out.wav", audio_arr, model.config.sampling_rate)
print("Saved indic_tts_out.wav")

Saved indic_tts_out.wav
